[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_04_data_science_math/ch18_data_wrangling_surveys/01_walkthrough.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 18 Walkthrough: Real-World Data Wrangling — Missing Data, Dirty Surveys, Joins, GroupBy, and Time Series

**Book Reference:** Chapter 18, Sections 18.1 – 18.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. All transforms follow **Pandas 3.x Copy-on-Write** best practices (`.copy()` + direct assignment, zero `inplace=True`).

In [ ]:
# @title Step 0: Install sharukh-tutor & Load Bundled Research Datasets
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import numpy as np
import pandas as pd
import sharukh_tutor as tutor

mch_raw = tutor.load_data("mch_survey_sample", as_pandas=True)
epi_raw = tutor.load_data("epi_surveillance", as_pandas=True)

print("MCH survey shape:", mch_raw.shape)
print("Epi panel shape: ", epi_raw.shape)


## Section 18.3.1 — Missingness Audit (MCAR / MAR / MNAR) & Dirty Survey Cleaning

In [ ]:
# 1. Audit explicit NaNs BEFORE sentinel cleaning
raw_counts = mch_raw.isna().sum().astype(int)
raw_pct = (raw_counts / len(mch_raw) * 100).round(2)
raw_audit = pd.DataFrame(
    {"missing_count": raw_counts, "missing_pct": raw_pct}
).sort_values("missing_pct", ascending=False)
print("Explicit NaNs before cleaning:")
print(raw_audit.head(3))

# 2. Inspect dirty casing and hidden DHS sentinel 999
raw_res = mch_raw["residence"].value_counts().to_dict()
raw_anc_mean = round(mch_raw["anc_visits"].mean(), 2)
print("\nRaw residence counts:", raw_res)
print("Raw anc_visits mean (with 999!):", raw_anc_mean)

# 3. Non-mutating Pandas 3.x cleaning copy
mch = mch_raw.copy()
mch["residence"] = mch["residence"].str.strip().str.title()
mch["anc_visits"] = mch["anc_visits"].replace(999, np.nan)
mch["maternal_hb_g_dl"] = pd.to_numeric(
    mch["maternal_hb_g_dl"], errors="coerce"
)

print("Clean residence counts:", mch["residence"].value_counts().to_dict())
print("Clean anc_visits NaNs: ", int(mch["anc_visits"].isna().sum()))
print("Clean anc_visits mean: ", round(mch["anc_visits"].mean(), 2))


## Section 18.3.2 — Group-Median Imputation with `.groupby(...).transform('median')`

In [ ]:
# Check missing Hb by residence (MAR diagnostic) and group medians
hb_na_by_res = mch.groupby("residence")["maternal_hb_g_dl"].apply(
    lambda s: int(s.isna().sum())
)
hb_med_by_res = mch.groupby("residence")["maternal_hb_g_dl"].median()
print("Missing Hb by residence:", hb_na_by_res.to_dict())
print("Median Hb by residence: ", hb_med_by_res.to_dict())

# Impute missing Hb with per-residence median (non-mutating copy)
mch_imp = mch.copy()
group_medians = (
    mch_imp.groupby("residence")["maternal_hb_g_dl"]
    .transform("median")
)
mch_imp["maternal_hb_g_dl"] = mch_imp["maternal_hb_g_dl"].fillna(
    group_medians
)

post_means = (
    mch_imp.groupby("residence")["maternal_hb_g_dl"]
    .mean().round(2).to_dict()
)
print("Remaining Hb NaNs:", int(mch_imp["maternal_hb_g_dl"].isna().sum()))
print("Post-imputation means:", post_means)


## Section 18.3.3 — Safe Relational Joins with `pd.merge(..., validate='one_to_one')`

In [ ]:
patients = pd.DataFrame(
    {
        "patient_id": ["P-101", "P-102", "P-103", "P-104"],
        "residence": ["Rural", "Urban", "Rural", "Urban"],
    }
)
labs = pd.DataFrame(
    {
        "patient_id": ["P-101", "P-102", "P-104", "P-999"],
        "ferritin_ng_ml": [18.5, 42.0, 31.2, 25.0],
    }
)

joined = pd.merge(
    patients, labs, on="patient_id", how="left", validate="one_to_one"
)
print(joined.to_string(index=False))


## Section 18.3.4 — Split-Apply-Combine with Named `.agg()`

In [ ]:
wealth_tbl = mch_imp.groupby("wealth_quintile").agg(
    n_households=("household_id", "count"),
    stunting_pct=("child_stunted", "mean"),
    anemia_pct=("maternal_anemia", "mean"),
    mean_anc=("anc_visits", "mean"),
)
wealth_tbl["stunting_pct"] = (wealth_tbl["stunting_pct"] * 100).round(1)
wealth_tbl["anemia_pct"] = (wealth_tbl["anemia_pct"] * 100).round(1)
wealth_tbl["mean_anc"] = wealth_tbl["mean_anc"].round(2)

order = ["Poorest", "Poorer", "Middle", "Richer", "Richest"]
print(wealth_tbl.reindex(order))


## Section 18.3.5 — Panel Time Series: Per-District Lags & Rolling Windows

In [ ]:
epi = epi_raw.sort_values(["district", "year", "epi_week"]).copy()
epi["prev_week_cases"] = epi.groupby("district")["dengue_cases"].shift(1)
epi["rolling_4w"] = (
    epi.groupby("district")["dengue_cases"]
    .transform(lambda s: s.rolling(window=4, min_periods=1).mean())
    .round(2)
)

cols = ["district", "year", "epi_week", "dengue_cases",
        "prev_week_cases", "rolling_4w"]
dhaka_top = epi[epi["district"] == "Dhaka"][cols].head(5)
ctg_top = epi[epi["district"] == "Chattogram"][cols].head(3)
print("Dhaka first 5 weeks:")
print(dhaka_top.to_string(index=False))
print("\nChattogram first 3 weeks (zero leakage from Dhaka!):")
print(ctg_top.to_string(index=False))


## Section 18.3.6 — Break It On Purpose! (Wrangling Error Anatomy)

In [ ]:
# Intentional Bug #1: MergeError — duplicate key caught by validate
dup_labs = pd.DataFrame(
    {
        "patient_id": ["P-101", "P-101", "P-102"],
        "ferritin_ng_ml": [18.5, 22.0, 42.0],
    }
)
pd.merge(
    patients, dup_labs, on="patient_id", how="left", validate="one_to_one"
)


In [ ]:
# Intentional Bug #2: KeyError — mismatched join key name
misnamed_labs = pd.DataFrame(
    {"household_id": ["P-101", "P-102"], "hb_g_dl": [11.4, 10.2]}
)
pd.merge(patients, misnamed_labs, on="patient_id", how="left")


In [ ]:
# Intentional Bug #3: ValueError — .astype(float) on dirty string 'N/A'
dirty_hb = pd.Series(["11.2", "N/A", "9.8"])
dirty_hb.astype(float)


## Section 18.4 — Stage 4 Blueprint: End-to-End Maternal Equity & Surveillance Pipeline

In [ ]:
def build_equity_brief(raw_mch):
    """Clean DHS survey and summarize residence x wealth equity gaps."""
    df = raw_mch.copy()
    df["residence"] = df["residence"].str.strip().str.title()
    df["anc_visits"] = df["anc_visits"].replace(999, np.nan)
    df["maternal_hb_g_dl"] = pd.to_numeric(
        df["maternal_hb_g_dl"], errors="coerce"
    )
    med = df.groupby("residence")["maternal_hb_g_dl"].transform("median")
    df["maternal_hb_g_dl"] = df["maternal_hb_g_dl"].fillna(med)
    df["anc_4plus"] = (df["anc_visits"] >= 4).fillna(False).astype(bool)

    brief = df.groupby(
        ["residence", "wealth_quintile"], as_index=False
    ).agg(
        n_households=("household_id", "count"),
        stunting_pct=("child_stunted", "mean"),
        anemia_pct=("maternal_anemia", "mean"),
        anc4_pct=("anc_4plus", "mean"),
    )
    for col in ("stunting_pct", "anemia_pct", "anc4_pct"):
        brief[col] = (brief[col] * 100).round(1)
    return brief.sort_values(["residence", "wealth_quintile"]).reset_index(
        drop=True
    )

equity_brief = build_equity_brief(mch_raw)
print(equity_brief.to_string(index=False))
